# Option C: Per-target base_rate adjustment

**Date:** 2026-04-18.
**Hypothesis:** Q1 over-prediction is driven by base_rate being target-invariant. A mass-market critic has base_rate≈0.85 whether the target is a blockbuster or a niche indie, but their real `P(review | target)` varies by movie type. Per-target adjustment scales each critic's contribution by how likely that critic tier is to engage with that target tier.

**Design (tiered lookup):**
1. **Critic tiers** (3): by cohort-wide review count. Tercile splits.
2. **Target tiers** (4): by `observed_count` at T-3d. Quartile splits.
3. **Multiplier matrix** `M[critic_tier, target_tier]` = `P(review | c_tier, t_tier) / P(review | c_tier)`. Computed from cohort.
4. **Inference:** for target with tier t, each critic's base_rate is scaled by `M[c_tier, t]`.

**Control:** weighted KDE at n=20 (best config so far, +7.7% cohort).
**Candidate:** weighted KDE at n=20 + per-target base_rate adjustment.

**Expected outcome:** Q1 over-prediction drops (mass-market critics get scaled down for low-volume targets). Other quartiles may or may not move. H/m outliers probably don't move much (their issue is cohort coverage).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_with_scores,
    snapshot_state, passes_skip_rules_for_snap,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    critic_activity_counts,
    compute_base_rate_multiplier_matrix, adjusted_predict_window,
    _assign_tier,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0

CACHE = CACHE_DIR / 'option_c_base_rate_adjustment.pkl'
print('Ready.')

## Step 1 — Compute tiers

**Critic tiers** (3): tercile splits on cohort-wide review counts.
**Target tiers** (4): quartile splits on observed_count at T-3d.

In [ ]:
# Critic tiers
activity = critic_activity_counts()
activity_arr = np.array(list(activity.values()))
critic_cuts = [float(np.percentile(activity_arr, 33)), float(np.percentile(activity_arr, 66))]
print(f'Critic tier cutoffs (activity = # movies reviewed): {critic_cuts}')
critic_tier_map = {
    c: _assign_tier(a, critic_cuts) for c, a in activity.items()
}
from collections import Counter
print('Critics per tier:', Counter(critic_tier_map.values()))
print()

# Compute observed_count at T-3d for each target in cohort
target_observed = {}
for target in close_date_map:
    target_gap = gap_for_slug(target)
    if target_gap is None:
        continue
    target_close = close_date_map[target]
    snap_time = target_close - pd.Timedelta(days=SNAP)
    state = snapshot_state(target, snap_time)
    passed, _ = passes_skip_rules_for_snap(state, SNAP)
    if not passed:
        continue
    target_observed[target] = state['observed_count']

# Target tier cutoffs from observed_count distribution
obs_arr = np.array(list(target_observed.values()))
target_cuts = [float(np.percentile(obs_arr, q)) for q in [25, 50, 75]]
print(f'Target tier cutoffs (observed_count at T-3d): {target_cuts}')
target_tier_map = {
    s: _assign_tier(obs, target_cuts) for s, obs in target_observed.items()
}
print('Targets per tier:', Counter(target_tier_map.values()))

## Step 2 — Compute multiplier matrix from cohort

In [ ]:
multiplier_raw, cell_rate, row_rate = compute_base_rate_multiplier_matrix(
    target_tier_map, critic_tier_map,
)
print('Empirical P(review) by (critic_tier, target_tier):')
print()
print(pd.DataFrame(cell_rate, index=['low', 'mid', 'high'],
                    columns=['T1_low', 'T2', 'T3', 'T4_high']).round(3))
print()
print('Row averages (critic_tier cohort rate):', dict(zip(['low','mid','high'], row_rate.round(3))))
print()
print('Raw multiplier matrix (P(review|c,t) / P(review|c)):')
print(pd.DataFrame(multiplier_raw, index=['low', 'mid', 'high'],
                    columns=['T1_low', 'T2', 'T3', 'T4_high']).round(2))
print()
# Asymmetric cap: only allow down-scaling. _compute_scaling already handles
# up-correction at the aggregate level, so allowing per-critic up-scaling
# double-counts and blows up high-volume targets.
multiplier = np.minimum(multiplier_raw, 1.0)
print('Capped multiplier matrix (down-only):')
print(pd.DataFrame(multiplier, index=['low', 'mid', 'high'],
                    columns=['T1_low', 'T2', 'T3', 'T4_high']).round(2))


## Step 3 — LOO at T-3d, comparing weighted vs weighted+adjusted

In [ ]:
def run_sweep(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    for i, target in enumerate(close_date_map):
        if target not in target_tier_map:
            continue
        target_gap = gap_for_slug(target)
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - SNAP
        if target_window_days <= 0:
            continue
        target_critics = state['observed_critics']
        target_tier = target_tier_map[target]

        scores = combined_score_with_scores(
            target, target_gap, target_critics, target_window_days,
            k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
        )
        if len(scores) < 5:
            continue

        mr = reviews[reviews['movie_slug'] == target].copy()
        mr['dbc'] = (target_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((mr['dbc'] > midnight_utc_dbc) & (mr['dbc'] <= SNAP)).sum())

        # Build weighted KDE (same for both configs)
        profiles = build_weighted_critic_profiles(reviews, close_date_map, scores, verbose=False)
        model = build_weighted_kde_lambda_model(
            profiles,
            bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
            bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )

        # Control: weighted KDE with unadjusted base_rates
        pred_ctrl = predict_window_custom(
            model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
            observed_critics=target_critics,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )

        # Candidate: weighted KDE + per-target base_rate adjustment
        pred_adj = adjusted_predict_window(
            model, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
            observed_critics=target_critics,
            target_tier=target_tier,
            critic_tier_map=critic_tier_map,
            multiplier_matrix=multiplier,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )

        rows.append({
            'target': target,
            'target_tier': target_tier,
            'target_gap': target_gap,
            'observed_count': state['observed_count'],
            'pred_ctrl': float(pred_ctrl),
            'pred_adj': float(pred_adj),
            'actual_phase1': actual_p1,
            'err_ctrl': float(pred_ctrl) - actual_p1,
            'err_adj': float(pred_adj) - actual_p1,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

results = run_sweep()
print(f'\nn={len(results)}')

## Aggregate and stratified MAE

In [ ]:
results['abs_err_ctrl'] = results['err_ctrl'].abs()
results['abs_err_adj'] = results['err_adj'].abs()
results['q_actual'] = pd.qcut(results['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

def summarize(df, label):
    mae_c = df['abs_err_ctrl'].mean()
    mae_a = df['abs_err_adj'].mean()
    me_c = df['err_ctrl'].mean()
    me_a = df['err_adj'].mean()
    delta_pct = 100 * (mae_c - mae_a) / mae_c if mae_c > 0 else 0
    print(f'{label:42s}  n={len(df):3d}  weighted={mae_c:6.2f}  +adj={mae_a:6.2f}  '
          f'delta={mae_c-mae_a:+6.2f}  {delta_pct:+6.1f}%  (me: {me_c:+5.2f} → {me_a:+5.2f})')

print('Weighted KDE vs Weighted KDE + per-target base_rate adjustment. Positive delta = adj better.\n')
summarize(results, 'Full cohort')
print()
for q in ['Q1','Q2','Q3','Q4']:
    sub = results[results['q_actual'] == q]
    if len(sub):
        lo, hi = int(sub['actual_phase1'].min()), int(sub['actual_phase1'].max())
        summarize(sub, f'{q} (actual [{lo}, {hi}])')

## By target volume tier (the signal we adjusted on)

In [ ]:
for t in [0, 1, 2, 3]:
    sub = results[results['target_tier'] == t]
    if len(sub):
        obs_lo = int(sub['observed_count'].min())
        obs_hi = int(sub['observed_count'].max())
        summarize(sub, f'Target tier {t} (observed_count [{obs_lo}, {obs_hi}])')

## H/m subset

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm = results[results['target'].isin(HM)].copy()
cols = ['target', 'target_tier', 'observed_count', 'actual_phase1',
        'pred_ctrl', 'pred_adj', 'err_ctrl', 'err_adj']
print('H/m subset:')
print(hm[cols].to_string(index=False, float_format='%.2f'))
print()
summarize(hm, 'H/m aggregate (5 movies)')

## Decision

- **Q1 improvement + cohort non-regression** → adjustment addresses Q1 over-prediction as hypothesized; ship alongside weighted KDE.
- **Q1 improves but cohort regresses** → over-correcting; needs smoother multipliers or wider stratification.
- **No change anywhere** → multipliers are all near 1.0, meaning tier-level variation is swamped by within-tier noise. Tier coarsening probably too aggressive.